# Identificador de Call Credit Spreads — v2

Melhorias em relacao ao v1:
- `symbol_info` reutilizado da fase de selecao (elimina round trips duplicados)
- `copy_rates_range` com janela de 5 dias (funciona com mercado aberto e fechado)
- Candle correto: `rates[-2]` com mercado aberto, `rates[-1]` com mercado fechado
- Geracao de spreads com pre-filtro de largura (evita loop O(n2) completo)
- Guard no bloco de metricas quando df_spreads vem vazio

In [1]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta
from IPython.display import display

if not mt5.initialize():
    raise RuntimeError(f'Falha ao inicializar MT5: {mt5.last_error()}')

print('MT5 conectado')
print(mt5.account_info())

MT5 conectado
AccountInfo(login=18901295, trade_mode=2, leverage=1, limit_orders=0, margin_so_mode=1, trade_allowed=True, trade_expert=True, margin_mode=2, currency_digits=2, fifo_close=False, balance=8937.0, credit=0.0, profit=-587.0, equity=8350.0, margin=4988.0, margin_free=3362.0, margin_level=167.401764234162, margin_so_call=0.0, margin_so_so=0.0, margin_initial=0.0, margin_maintenance=0.0, assets=0.0, liabilities=0.0, commission_blocked=0.0, name='RAFAEL LANZA CARIOCA', server='XPMT5-PRD', currency='BRL', company='XP Investimentos CCTVM S/A')


In [2]:
# ==============================================================================
# PARAMETROS — edite aqui
# ==============================================================================

UNDERLYINGS_BASE = [
    'PETR',
    'VALE',
    'ITUB',
    'BBDC',
    'BBAS',
    'ABEV',
    'WEGE',
    'B3SA',
    'SUZB',
]

MIN_SELL_VOLUME  = 5_000
MIN_BUY_VOLUME   = 1_000
MIN_OTM_PCT      = 2.0
MIN_RETURN_PCT   = 30.0
MIN_CREDIT       = 0.50
MIN_SPREAD_WIDTH = 1.0
MAX_SPREAD_WIDTH = 2.0

In [3]:
# ==============================================================================
# HELPERS
# ==============================================================================

def mercado_aberto():
    """
    True se B3 estiver aberta agora (seg-sex 10h-17h55, horario local).
    """
    agora = datetime.now()
    if agora.weekday() >= 5:
        return False
    minutos = agora.hour * 60 + agora.minute
    return (10 * 60) <= minutos <= (17 * 60 + 55)


MERCADO_ABERTO = mercado_aberto()
# Com mercado aberto rates[-1] e o candle parcial do dia — usamos rates[-2].
# Com mercado fechado rates[-1] ja e o ultimo fechamento real.
CANDLE_IDX = -2 if MERCADO_ABERTO else -1
print(f'Mercado: {"ABERTO" if MERCADO_ABERTO else "FECHADO"}  |  candle idx: {CANDLE_IDX}')


def classify_type(info):
    """Classificacao canonica XP/B3 via path."""
    if info is None or info.path is None:
        return 'UNKNOWN'
    path = info.path.upper()
    if 'OPCOES' in path:
        if info.option_right == 0:
            return 'CALL'
        elif info.option_right == 1:
            return 'PUT'
        return 'OPTION'
    return 'SPOT'


def get_close_and_volume(ticker):
    """
    Retorna (fechamento, volume) para uso como preco de referencia.
    Mesma logica do monitor de travas:
      1. tick.last  — ultimo preco negociado (funciona com mercado fechado)
      2. candle D-1 — fallback historico com janela de 15 dias
    Volume sempre vem do candle D-1 pois tick nao tem volume acumulado.
    """
    mt5.symbol_select(ticker, True)

    # tenta ultimo preco negociado primeiro
    tick = mt5.symbol_info_tick(ticker)
    preco_tick = float(tick.last) if tick and tick.last > 0 else np.nan

    # candle para volume (e fallback de preco)
    rates = mt5.copy_rates_range(
        ticker,
        mt5.TIMEFRAME_D1,
        datetime.utcnow() - timedelta(days=15),
        datetime.utcnow(),
    )

    fechamento_candle = np.nan
    volume            = np.nan

    if rates is not None and len(rates) >= 2:
        candle            = rates[CANDLE_IDX]
        fechamento_candle = float(candle['close'])
        volume            = int(candle['real_volume']) if candle['real_volume'] > 0 else int(candle['tick_volume'])

    # preco final: tick.last se disponivel, senao fechamento do candle
    fechamento = preco_tick if not np.isnan(preco_tick) else fechamento_candle

    return fechamento, volume


def third_friday(year, month):
    """Retorna a data da 3a sexta-feira do mes/ano informado."""
    d = datetime(year, month, 1)
    days_until_friday = (4 - d.weekday()) % 7
    first_friday = d + timedelta(days=days_until_friday)
    return (first_friday + timedelta(days=14)).date()


UNDERLYING_FALLBACK = {
    'VALE': 'VALE3',
    'BOVA': 'BOVA11',
    'SMAL': 'SMAL11',
    'IVVB': 'IVVB11',
}


def infer_underlying(ticker, description):
    """Infere o ticker do ativo objeto a partir da opcao."""
    base = ticker[:4]
    if isinstance(description, str):
        d = description.upper()
        if re.search(r'\bPN\b', d):
            return f'{base}4'
        if re.search(r'\bON\b', d):
            return f'{base}3'
    if base in UNDERLYING_FALLBACK:
        return UNDERLYING_FALLBACK[base]
    return f'{base}3'


print('Helpers carregados')

Mercado: FECHADO  |  candle idx: -1
Helpers carregados


In [4]:
# ==============================================================================
# DATAS DE VENCIMENTO
# ==============================================================================

now = datetime.utcnow()

current_expiry_date = third_friday(now.year, now.month)

next_month = now.month + 1 if now.month < 12 else 1
next_year  = now.year if now.month < 12 else now.year + 1
next_expiry_date = third_friday(next_year, next_month)

print(f'Vencimento atual : {current_expiry_date}')
print(f'Proximo vencimento: {next_expiry_date}')

Vencimento atual : 2026-04-17
Proximo vencimento: 2026-05-15


In [5]:
# ==============================================================================
# SELECAO DE TICKERS
# symbol_info guardado aqui para reutilizar no bloco seguinte
# ==============================================================================

SPOT_RE  = re.compile(r'^[A-Z]{4}(3|4|11)$')
symbols  = mt5.symbols_get() or []
selected = []  # lista de dicts: ticker, info, asset_type

for s in symbols:
    name   = s.name.upper()
    path   = getattr(s, 'path', '') or ''
    if not path:
        continue
    if not any(name.startswith(u) for u in UNDERLYINGS_BASE):
        continue

    path_up = path.upper()

    # acoes a vista
    if 'VISTA' in path_up:
        if SPOT_RE.match(name):
            mt5.symbol_select(s.name, True)
            info = mt5.symbol_info(s.name)
            selected.append({'ticker': s.name, 'info': info, 'asset_type': 'SPOT'})
        continue

    # opcoes — filtra por vencimento e tipo CALL
    if 'OPCO' in path_up:
        mt5.symbol_select(s.name, True)
        info = mt5.symbol_info(s.name)
        if not info or not info.expiration_time or info.expiration_time <= 0:
            continue
        try:
            exp_date = datetime.fromtimestamp(info.expiration_time).date()
        except Exception:
            continue
        if exp_date not in (current_expiry_date, next_expiry_date):
            continue
        if classify_type(info) != 'CALL':
            continue
        selected.append({'ticker': s.name, 'info': info, 'asset_type': 'CALL'})

n_spots = sum(1 for x in selected if x['asset_type'] == 'SPOT')
n_calls = sum(1 for x in selected if x['asset_type'] == 'CALL')
print(f'{len(selected)} tickers selecionados  ({n_spots} spots, {n_calls} calls)')

2883 tickers selecionados  (12 spots, 2871 calls)


In [6]:
# ==============================================================================
# BUSCA FECHAMENTO E VOLUME
# symbol_info reutilizado — apenas copy_rates_range por ticker
# ==============================================================================

rows = []

for item in selected:
    ticker     = item['ticker']
    info       = item['info']
    asset_type = item['asset_type']

    strike = (
        float(info.option_strike)
        if info and info.option_strike and info.option_strike > 0
        else np.nan
    )

    expiration = pd.NaT
    try:
        if info and info.expiration_time and info.expiration_time > 0:
            expiration = datetime.fromtimestamp(info.expiration_time)
    except Exception:
        pass

    underlying = (
        ticker
        if asset_type == 'SPOT'
        else infer_underlying(ticker, info.description if info else None)
    )

    fechamento, volume = get_close_and_volume(ticker)

    rows.append({
        'ticker'       : ticker,
        'type'         : asset_type,
        'underlying'   : underlying,
        'strike'       : strike,
        'expiration'   : expiration,
        'fechamento_d1': fechamento,
        'volume_d1'    : volume,
    })

df_ativos = pd.DataFrame(rows)
print(f'df_ativos: {len(df_ativos)} linhas')

# diagnostico: spots sem fechamento indicam problema de dados
spots_sem_preco = df_ativos[(df_ativos['type'] == 'SPOT') & df_ativos['fechamento_d1'].isna()]
if not spots_sem_preco.empty:
    print(f'Aviso: {len(spots_sem_preco)} spot(s) sem fechamento: {spots_sem_preco["ticker"].tolist()}')

display(df_ativos.head(10))

df_ativos: 2883 linhas
Aviso: 1 spot(s) sem fechamento: ['BBAS11']


,ticker,type,underlying,strike,expiration,fechamento_d1,volume_d1
0,BBAS11,SPOT,BBAS11,NaN,NaT,NaN,NaN
1,BBAS3,SPOT,BBAS3,NaN,NaT,23.43,NaN
2,BBDC3,SPOT,BBDC3,NaN,NaT,16.77,NaN
3,BBDC4,SPOT,BBDC4,NaN,NaT,19.14,NaN
4,ITUB3,SPOT,ITUB3,NaN,NaT,42.77,NaN
5,ITUB4,SPOT,ITUB4,NaN,NaT,43.30,NaN
6,PETR3,SPOT,PETR3,NaN,NaT,52.95,NaN
7,PETR4,SPOT,PETR4,NaN,NaT,48.10,NaN
8,SUZB3,SPOT,SUZB3,NaN,NaT,50.78,NaN
9,VALE3,SPOT,VALE3,NaN,NaT,83.69,NaN


In [7]:
# ==============================================================================
# SEPARA SPOTS E CALLS — mapeia preco do spot
# ==============================================================================

df_spot  = df_ativos[df_ativos['type'] == 'SPOT'].copy()
df_calls = df_ativos[df_ativos['type'] == 'CALL'].copy()

spot_price_map         = dict(zip(df_spot['ticker'], df_spot['fechamento_d1']))
df_calls['spot_price'] = df_calls['underlying'].map(spot_price_map)

# calls OTM validas (strike acima do spot, preco e spot disponiveis)
df_calls_otm = df_calls[
    df_calls['strike'].notna() &
    df_calls['spot_price'].notna() &
    (df_calls['strike'] > df_calls['spot_price']) &
    df_calls['fechamento_d1'].notna()
].copy()

print(f'Calls OTM validas: {len(df_calls_otm)}')

Calls OTM validas: 598


In [8]:
# ==============================================================================
# GERA CALL CREDIT SPREADS
# Pre-filtro de largura elimina combinacoes invalidas antes do loop interno
# ==============================================================================

if df_calls_otm.empty:
    print('Nenhuma call OTM valida encontrada. Verifique se o mercado tem dados para as datas de vencimento.')
    df_spreads = pd.DataFrame()
else:
    spreads = []

    for (underlying, expiration), g in df_calls_otm.groupby(['underlying', 'expiration']):
        g = g.sort_values('strike').reset_index(drop=True)

        for _, sell in g.iterrows():
            buy_min    = sell['strike'] + MIN_SPREAD_WIDTH
            buy_max    = sell['strike'] + MAX_SPREAD_WIDTH
            candidates = g[(g['strike'] >= buy_min) & (g['strike'] <= buy_max)]

            for _, buy in candidates.iterrows():
                credit = sell['fechamento_d1'] - buy['fechamento_d1']
                if credit <= 0:
                    continue
                spreads.append({
                    'underlying'  : underlying,
                    'expiration'  : expiration,
                    'sell_ticker' : sell['ticker'],
                    'buy_ticker'  : buy['ticker'],
                    'sell_strike' : sell['strike'],
                    'buy_strike'  : buy['strike'],
                    'spot_price'  : sell['spot_price'],
                    'sell_premium': sell['fechamento_d1'],
                    'buy_premium' : buy['fechamento_d1'],
                    'credit'      : credit,
                    'sell_volume' : sell['volume_d1'],
                    'buy_volume'  : buy['volume_d1'],
                })

    df_spreads = pd.DataFrame(spreads)
    print(f'{len(df_spreads)} pares gerados antes dos filtros')

1117 pares gerados antes dos filtros


In [9]:
# ==============================================================================
# METRICAS + FILTROS
# ==============================================================================

if df_spreads.empty:
    print('df_spreads vazio — nenhum spread para filtrar.')
    df_filt = pd.DataFrame()
else:
    df = df_spreads.copy()

    df['spread_width'] = df['buy_strike'] - df['sell_strike']
    df['otm_pct']      = (df['sell_strike'] - df['spot_price']) / df['spot_price'] * 100
    df['max_loss']     = df['spread_width'] - df['credit']
    df['return_pct']   = df['credit'] / df['max_loss'] * 100
    df['exp_date']     = df['expiration'].dt.date

    df_filt = df[
        (df['sell_volume']  >= MIN_SELL_VOLUME)  &
        (df['buy_volume']   >= MIN_BUY_VOLUME)   &
        (df['otm_pct']      >= MIN_OTM_PCT)      &
        (df['return_pct']   >= MIN_RETURN_PCT)   &
        (df['credit']       >= MIN_CREDIT)       &
        (df['spread_width'] >= MIN_SPREAD_WIDTH) &
        (df['spread_width'] <= MAX_SPREAD_WIDTH) &
        (df['max_loss']     >  0)
    ].copy()

    print(f'{len(df_filt)} spreads apos filtros')

0 spreads apos filtros


In [10]:
# ==============================================================================
# RESULTADOS — MES ATUAL E PROXIMO
# ==============================================================================

cols_exibir = [
    'underlying', 'expiration',
    'sell_ticker', 'buy_ticker',
    'sell_strike', 'buy_strike',
    'spot_price', 'sell_premium', 'buy_premium',
    'credit', 'spread_width', 'otm_pct', 'max_loss', 'return_pct',
    'sell_volume', 'buy_volume',
]


def show_table(df_input, titulo):
    print('')
    print('=' * 60)
    print(titulo)
    print('=' * 60)
    if df_input.empty:
        print('Nenhuma trava encontrada.')
        return
    df_rank = df_input.sort_values(
        ['return_pct', 'otm_pct'],
        ascending=[False, False]
    )
    display(df_rank[cols_exibir].head(20))


if not df_filt.empty:
    df_current = df_filt[df_filt['exp_date'] == current_expiry_date]
    df_next    = df_filt[df_filt['exp_date'] == next_expiry_date]
    show_table(df_current, f'TRAVAS -- VENCIMENTO MES ATUAL ({current_expiry_date})')
    show_table(df_next,    f'TRAVAS -- PROXIMO VENCIMENTO ({next_expiry_date})')
else:
    print('Nenhum spread passou os filtros.')

Nenhum spread passou os filtros.


In [11]:
mt5.shutdown()
print('MT5 desconectado')

MT5 desconectado
